# 🚨 Detección de Anomalías Operativas mediante Densidad (DBSCAN)
## Módulo 4 · Outliers en Telemetría SCADA · Capacitación SLB

### Objetivos de la sesión:
1. Comparar la susceptibilidad de K-Means ante valores extremos frente a la inmunidad de DBSCAN.
2. Calibrar de forma matemática el radio de densidad Epsilon usando la distancia a vecinos más cercanos.
3. Aislar datos clasificados como ruido (-1) y graficarlos sobre los registros de producción diarios.

## ¿Qué librerías usaremos para algoritmos de densidad y vecinos más cercanos?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid")

# 1. Preparación de Telemetría

## ¿Cómo descargamos y cargamos los datos de telemetría SCADA `telemetria_anomala.csv` desde GitHub?

In [ ]:
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_4_no_supervisado/data/telemetria_anomala.csv -O telemetria_anomala.csv

df_scada = pd.read_csv('telemetria_anomala.csv')
df_scada.head()

## ¿Cómo visualizamos la distribución conjunta de presiones y caudales para identificar anomalías espaciales?

In [ ]:
# TU CÓDIGO AQUÍ: Genera un scatterplot de WHP_psi vs Qo_bpd para observar outliers dispersos
sns.scatterplot(data=df_scada, x='WHP_psi', y='Qo_bpd', color='red', alpha=0.6)
plt.title('Distribución Conjunta de Presión en Cabeza vs Caudal')
plt.show()

## ¿Cómo escalamos los datos para preparar el cálculo de densidad espacial en DBSCAN?

In [ ]:
features = ['Qo_bpd', 'WHP_psi']
# TU CÓDIGO AQUÍ: Escala las variables del DataFrame usando StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_scada[features])

# 2. Calibración de Epsilon

## ¿Cómo calculamos las distancias a los vecinos más cercanos para encontrar la 'rodilla' óptima de Epsilon?

In [ ]:
neighbors = NearestNeighbors(n_neighbors=4)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

# TU CÓDIGO AQUÍ: Ordena las distancias de menor a mayor y grafícalas para detectar la rodilla
distances_sorted = np.sort(distances[:, 3])

plt.plot(distances_sorted)
plt.axhline(y=0.25, color='r', linestyle='--', label='Epsilon recomendado (0.25)')
plt.title('Gráfico de Distancia al 4to Vecino Más Cercano')
plt.legend()
plt.show()

# 3. Entrenamiento y Aislamiento de Ruido

## ¿Cómo entrenamos DBSCAN con los parámetros optimizados (Eps=0.25, MinPts=4) e identificamos el ruido (-1)?

In [ ]:
# TU CÓDIGO AQUÍ: Instancia y entrena DBSCAN con eps=0.25 y min_samples=4
dbscan = DBSCAN(eps=0.25, min_samples=4)
df_scada['Cluster'] = dbscan.fit_predict(X_scaled)

# Contar puntos en cada cluster (recuerda que -1 representa el ruido / fallas SCADA)
df_scada['Cluster'].value_counts()

## ¿Cómo visualizamos en un gráfico bidimensional los datos limpios versus las fallas del sensor (ruido)?

In [ ]:
# TU CÓDIGO AQUÍ: Grafica un scatterplot de WHP vs Qo coloreando según la columna 'Cluster'
sns.scatterplot(data=df_scada, x='WHP_psi', y='Qo_bpd', hue='Cluster', palette='tab10', alpha=0.8)
plt.title('Detección Multivariada de Anomalías SCADA con DBSCAN')
plt.show()

> **🔍 Observación:** Los puntos clasificados con el color del grupo `-1` representan anomalías severas de telemetría (desacoplamiento de sensor, congelamiento de lectura o saltos erróneos), aislados sin contaminar el cluster principal sano.